#  Détection d'Objets avec DETR — Fine-tuning sur un Dataset Animalier

---

##  Description du projet

Ce notebook présente un pipeline complet de **détection d'objets** basé sur le modèle **DETR** (*DEtection TRansformer*, Facebook AI Research), appliqué à un dataset personnalisé d'images d'animaux au format **COCO**.

### Objectif
Fine-tuner un modèle DETR pré-entraîné (`facebook/detr-resnet-50`) pour détecter et localiser plusieurs espèces animales dans des images, en adaptant sa tête de classification au nombre de catégories du dataset.

### Architecture DETR
DETR est un détecteur d'objets basé sur les **Transformers** qui traite la détection comme un problème de prédiction d'ensembles (*set prediction*). Il élimine le besoin d'ancres (*anchors*) et de NMS (*Non-Maximum Suppression*) grâce à un mécanisme d'attention multi-têtes et un algorithme d'assignation bipartite (Hungarian matching).

---

##  Structure du notebook

| Étape | Description |
|-------|-------------|
| **1. Préparation des données** | Montage Drive, extraction du dataset |
| **2. Exploration du dataset** | Analyse des images et annotations COCO |
| **3. Analyse statistique** | Distribution des classes, des boîtes, qualité |
| **4. Nettoyage des données** | Suppression des anomalies, correction des boîtes |
| **5. Préparation pour DETR** | Dataset PyTorch, DataLoaders |
| **6. Configuration & Entraînement** | Fine-tuning, callbacks, sauvegarde |
| **7. Évaluation & Inférence** | Courbes d'apprentissage, chargement du modèle |

---

##  Prérequis

- **Environnement** : Google Colab avec GPU (T4 ou supérieur recommandé)
- **Stockage** : Google Drive avec `ena24.zip` et `annotations.json`
- **Librairies** : `transformers`, `torch`, `torchvision`, `datasets`, `accelerate`


---

## 1 Préparation de l'environnement et des données

### 1.1 Montage de Google Drive

On commence par monter Google Drive afin d'accéder aux fichiers du projet (images, annotations, checkpoints). Le dataset est stocké sous forme compressée (`.zip`) pour faciliter le transfert.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Extraction du dataset

Le dataset `ena24.zip` est extrait dans l'environnement Colab local (`/content/ena24`). Cette étape est nécessaire car les opérations d'I/O sur Colab sont beaucoup plus rapides en local que depuis Drive lors de l'entraînement.

> ** Structure attendue** : Le ZIP doit contenir les fichiers images directement (`.jpg`, `.png`, etc.).


In [ ]:
import zipfile
import os
import shutil

# Chemin vers le ZIP dans Google Drive
zip_path = "/content/drive/MyDrive/ena24.zip"   # adaptez le nom/dossier

# Dossier de destination dans Colab
extract_path = "/content/ena24"

# Créer le dossier de destination
os.makedirs(extract_path, exist_ok=True)

# Décompresser
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction terminée.")

### 1.3 Inventaire des images disponibles

On parcourt récursivement le dossier extrait pour recenser toutes les images, quels que soient leur sous-dossier ou extension.


In [ ]:
import glob

# Récupérer tous les fichiers image
image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tiff"]
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(extract_path, "**", ext), recursive=True))

print(f"Nombre d'images trouvées : {len(image_paths)}")

### 1.4 Visualisation d'un échantillon aléatoire

Avant toute manipulation, il est important de **visualiser quelques images** pour vérifier la qualité du dataset (résolution, luminosité, diversité des prises de vue).


In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

# Nombre d'images à afficher
n = 5
if len(image_paths) < n:
    n = len(image_paths)

random_images = random.sample(image_paths, n)

# Affichage dans une grille
fig, axes = plt.subplots(1, n, figsize=(15, 5))
if n == 1:
    axes = [axes]

for ax, img_path in zip(axes, random_images):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(img_path), fontsize=8)

plt.tight_layout()
plt.show()

---

## 2 Chargement et exploration des annotations COCO

### 2.1 Chargement du fichier JSON

Les annotations sont au **format COCO**, un standard largement utilisé en vision par ordinateur. Le fichier JSON contient trois listes principales :

| Clé | Description |
|-----|-------------|
| `images` | Métadonnées de chaque image (id, nom, dimensions) |
| `annotations` | Boîtes englobantes et labels (format `[x, y, w, h]`) |
| `categories` | Liste des classes d'objets avec leur identifiant |


In [ ]:
import json
import os
from google.colab import drive
drive.mount('/content/drive')

# Chemin vers le fichier JSON (exemple)
json_path = "/content/drive/MyDrive/annotations.json"

# Charger le JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# Afficher la structure générale
print("Type de data:", type(data))
print("Clés principales:", data.keys() if isinstance(data, dict) else "C'est une liste")

### 2.2 Inspection de la structure

On affiche la structure interne du JSON pour vérifier que toutes les clés attendues sont présentes et que les données sont correctement formatées.


In [ ]:
if isinstance(data, dict):
    # Voir les métadonnées
    for key in data.keys():
        print(f"\n--- {key} : {type(data[key])} ---")
        if isinstance(data[key], list) and len(data[key]) > 0:
            print(f"  Premier élément : {data[key][0]}")
        elif isinstance(data[key], dict):
            print(f"  Clés : {list(data[key].keys())[:5]}")

### 2.3 Statistiques générales du dataset

On extrait les trois listes principales et on affiche un résumé quantitatif du dataset : nombre d'images, d'annotations et de catégories.


In [ ]:
# Extraire les listes
images = data.get('images', [])
annotations = data.get('annotations', [])
categories = data.get('categories', [])

print(f"Nombre d'images : {len(images)}")
print(f"Nombre d'annotations : {len(annotations)}")
print(f"Nombre de catégories : {len(categories)}")

# Afficher les noms de catégories
print("\nCatégories :")
for cat in categories[:10]:
    print(f"  id {cat['id']} -> {cat['name']}")

### 2.4 Visualisation d'une annotation avec sa boîte englobante

On sélectionne une annotation aléatoire et on affiche l'image correspondante avec sa **boîte englobante** (bounding box) en rouge, ainsi que le nom de la catégorie.

> ** Format COCO** : `[x_min, y_min, width, height]` — les coordonnées sont en pixels absolus.


In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Construire un dictionnaire image_id -> nom_fichier
image_id_to_filename = {img['id']: img['file_name'] for img in images}

# Choisir une annotation aléatoire
ann = random.choice(annotations)
img_id = ann['image_id']
bbox = ann['bbox']  # [x, y, width, height] au format COCO
category_id = ann['category_id']

# Trouver le nom de la catégorie
cat_name = next((cat['name'] for cat in categories if cat['id'] == category_id), "inconnu")

# Chemin complet de l'image
img_filename = image_id_to_filename.get(img_id)
img_path = os.path.join("/content/ena24", img_filename)  # adapter le dossier racine

# Charger et afficher
img = Image.open(img_path)
fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(img)

# Dessiner le rectangle
x, y, w, h = bbox
rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
ax.add_patch(rect)
ax.set_title(f"{cat_name} (id {category_id})")
ax.axis('off')
plt.show()

---

## 3 Analyse exploratoire et statistique (EDA)

Cette section vise à **comprendre en profondeur le dataset** avant tout entraînement. Une EDA rigoureuse permet d'identifier des déséquilibres de classes, des anomalies ou des problèmes de qualité qui pourraient nuire aux performances du modèle.

### 3.1 Vérification de la cohérence image/annotation

On vérifie que toutes les images référencées dans le JSON sont bien présentes sur le disque. Des images manquantes provoqueraient des erreurs lors du chargement du dataset.


In [ ]:
import os
image_dir = "/content/ena24"
missing = []
for img in images:
    fname = img['file_name']
    full_path = os.path.join(image_dir, fname)
    if not os.path.exists(full_path):
        missing.append(fname)

print(f"Images manquantes : {len(missing)}")
if missing:
    print(missing[:5])

### 3.2 Conversion en DataFrames Pandas

On convertit les listes JSON en `DataFrame` Pandas pour faciliter les analyses statistiques, les filtres et les jointures.


In [ ]:
import pandas as pd

df_images = pd.DataFrame(images)
df_annotations = pd.DataFrame(annotations)
df_categories = pd.DataFrame(categories)

print("Aperçu des images :")
print(df_images.head())
print("\nAperçu des annotations :")
print(df_annotations.head())

### 3.3 Rechargement et initialisation des librairies d'analyse

Import des librairies nécessaires à l'analyse statistique approfondie.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Chargement (si pas déjà fait)
with open(json_path, 'r') as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

In [ ]:
# Conversion en DataFrames
df_img = pd.DataFrame(images)
df_ann = pd.DataFrame(annotations)
df_cat = pd.DataFrame(categories)

### 3.4 Distribution du nombre d'annotations par image

On analyse combien d'annotations (objets) chaque image contient. Un dataset déséquilibré (beaucoup d'images avec un seul objet vs quelques-unes avec des dizaines) peut influencer la stratégie d'entraînement.


In [ ]:
ann_per_image = df_ann.groupby('image_id').size().reset_index(name='nb_ann')
print("Statistiques du nombre d'annotations par image :")
print(ann_per_image['nb_ann'].describe())

### 3.5 Images sans annotation

Les images sans annotation sont inutilisables pour l'entraînement en détection d'objets. On les identifie ici pour décider si on les exclut ou si elles servent de données négatives.


In [ ]:
images_with_ann = set(df_ann['image_id'].unique())
all_image_ids = set(df_img['id'])
images_without_ann = all_image_ids - images_with_ann
print(f"\nImages sans annotation : {len(images_without_ann)} / {len(all_image_ids)}")

### 3.6 Distribution des catégories

On analyse la **distribution des classes** dans le dataset. Un fort déséquilibre entre catégories (class imbalance) peut conduire le modèle à privilégier les classes majoritaires au détriment des classes rares.


In [ ]:
cat_counts = df_ann['category_id'].value_counts().sort_index()
# Ajouter les noms
cat_counts.name = 'nb_annotations'
cat_dist = pd.DataFrame({'category_id': cat_counts.index,
                         'category_name': [df_cat[df_cat['id']==cid]['name'].values[0] for cid in cat_counts.index],
                         'count': cat_counts.values})
print("\nDistribution des catégories :")
print(cat_dist)

### 3.7 Visualisation de la distribution par catégorie

Un graphique en barres permet de visualiser rapidement le déséquilibre de classes. Les catégories sous-représentées pourront nécessiter des techniques d'augmentation de données ou de rééchantillonnage.


In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=cat_dist, x='category_name', y='count')
plt.xticks(rotation=90)
plt.title("Nombre d'annotations par catégorie")
plt.tight_layout()
plt.show()

### 3.8 Analyse des dimensions des boîtes englobantes

On analyse la taille des boîtes (largeur, hauteur, surface). Cela permet de détecter des objets très petits ou très grands et d'adapter les hyperparamètres du modèle (ex. seuil de détection, taille d'entrée).


In [ ]:
df_ann['bbox_width'] = df_ann['bbox'].apply(lambda b: b[2])
df_ann['bbox_height'] = df_ann['bbox'].apply(lambda b: b[3])
df_ann['bbox_area'] = df_ann['bbox_width'] * df_ann['bbox_height']

print("\nStatistiques des dimensions des boîtes :")
print(df_ann[['bbox_width', 'bbox_height', 'bbox_area']].describe())

### 3.9 Distribution visuelle des dimensions et surfaces

Les histogrammes révèlent la distribution réelle des tailles d'objets dans le dataset, information clé pour évaluer la difficulté de la tâche de détection.


In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.hist(df_ann['bbox_width'], bins=50, alpha=0.7, label='Largeur')
plt.hist(df_ann['bbox_height'], bins=50, alpha=0.7, label='Hauteur')
plt.xlabel('Pixels')
plt.ylabel('Fréquence')
plt.legend()
plt.title('Distribution des dimensions')
plt.subplot(1,2,2)
plt.hist(df_ann['bbox_area'], bins=50, color='green', alpha=0.7)
plt.xlabel('Surface (pixels²)')
plt.title('Distribution des surfaces')
plt.tight_layout()
plt.show()

---

## 4 Contrôle qualité et nettoyage des données

Un dataset de mauvaise qualité est l'une des principales causes d'échec d'un modèle. Cette section identifie et corrige les problèmes fréquents dans les datasets de détection.

### 4.1 Détection des annotations dupliquées

Des annotations dupliquées (même image, même boîte, même classe) peuvent biaiser l'apprentissage. On les identifie et les supprime.


In [ ]:
# Copie de sécurité
df_ann_temp = df_ann.copy()
# Convertir bbox en tuple
df_ann_temp['bbox_tuple'] = df_ann_temp['bbox'].apply(tuple)

# Maintenant on peut chercher les doublons
dup_ann = df_ann_temp.duplicated(subset=['image_id', 'bbox_tuple', 'category_id']).sum()
print(f"Annotations dupliquées (même image, même boîte, même classe) : {dup_ann}")

# Optionnel : supprimer ces doublons
if dup_ann > 0:
    df_ann_clean = df_ann_temp.drop_duplicates(subset=['image_id', 'bbox_tuple', 'category_id']).drop('bbox_tuple', axis=1)
    print(f"Après suppression : {len(df_ann_clean)} annotations")
else:
    df_ann_clean = df_ann

### 4.2 Détection des boîtes englobantes invalides

Une boîte est invalide si ses coordonnées sont négatives ou si ses dimensions sont nulles ou négatives. Ces cas doivent être filtrés avant l'entraînement car ils provoquent des erreurs de calcul de la loss.


In [ ]:
invalid_bbox = df_ann[
    (df_ann['bbox'].apply(lambda b: b[0] < 0)) |
    (df_ann['bbox'].apply(lambda b: b[1] < 0)) |
    (df_ann['bbox'].apply(lambda b: b[2] <= 0)) |
    (df_ann['bbox'].apply(lambda b: b[3] <= 0))
]
print(f"Boîtes invalides : {len(invalid_bbox)}")

### 4.3 Détection des boîtes hors limites

Les boîtes qui dépassent les bords de l'image (`x + w > image_width` ou `y + h > image_height`) sont problématiques. On les détecte en fusionnant les annotations avec les métadonnées d'images.


In [ ]:
# Joindre les dimensions de l'image
df_ann_with_img = df_ann.merge(df_img[['id', 'width', 'height']], left_on='image_id', right_on='id')
out_of_bounds = df_ann_with_img[
    (df_ann_with_img['bbox'].apply(lambda b: b[0] + b[2] > df_ann_with_img['width'])) |
    (df_ann_with_img['bbox'].apply(lambda b: b[1] + b[3] > df_ann_with_img['height']))
]
print(f"Boîtes débordant de l'image : {len(out_of_bounds)}")

### 4.4 Images avec le plus grand nombre d'annotations

On identifie les images les plus denses en annotations. Ces images complexes peuvent être des cas difficiles pour le modèle.


In [ ]:
top_multi = ann_per_image.nlargest(10, 'nb_ann')
top_multi = top_multi.merge(df_img[['id','file_name']], left_on='image_id', right_on='id')
print(top_multi)

### 4.5 Rechargement propre des données

Rechargement du fichier JSON et conversion en DataFrames pour repartir sur une base cohérente avant le filtrage.


In [ ]:
# Recharger le JSON (si pas déjà en mémoire)
import json
with open("/content/drive/MyDrive/annotations.json", 'r') as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

df_img = pd.DataFrame(images)        # colonnes : id, file_name, width, height
df_ann = pd.DataFrame(annotations)   # colonnes : id, image_id, category_id, bbox
df_cat = pd.DataFrame(categories)    # colonnes : name, id

### 4.6 Filtrage des images et annotations exploitables

On filtre le dataset pour ne conserver que les images physiquement présentes sur le disque, et les annotations qui leur correspondent. C'est une étape critique avant la construction du dataset PyTorch.


In [ ]:
import os
image_dir = "/content/ena24"  # votre dossier d'images extraites
existing_files = set(os.listdir(image_dir))

df_img['file_exists'] = df_img['file_name'].apply(lambda f: f in existing_files)
df_img_filtered = df_img[df_img['file_exists']].copy()  # garde toutes les colonnes
print(f"Images exploitables : {len(df_img_filtered)}")
print(df_img_filtered[['id', 'file_name']].head())  # vérifier la présence de file_name

valid_ids = set(df_img_filtered['id'])
df_ann_filtered = df_ann[df_ann['image_id'].isin(valid_ids)].copy()
print(f"Annotations conservées : {len(df_ann_filtered)}")

### 4.7 Vérification des boîtes hors limites sur le dataset filtré


In [ ]:
# Fusionner avec les dimensions de l'image
df_merged = df_ann_filtered.merge(
    df_img_filtered[['id', 'width', 'height', 'file_name']],
    left_on='image_id',
    right_on='id'
)

# Vérification ligne par ligne
def is_out_of_bounds(row):
    x, y, w, h = row['bbox']
    return (x + w > row['width']) or (y + h > row['height'])

df_merged['out_of_bounds'] = df_merged.apply(is_out_of_bounds, axis=1)
out_count = df_merged['out_of_bounds'].sum()
print(f"Boîtes hors limites : {out_count} / {len(df_merged)}")

# Afficher quelques exemples si besoin
if out_count > 0:
    print(df_merged[df_merged['out_of_bounds']][['file_name', 'bbox', 'width', 'height']].head())

### 4.8 Top 10 des images les plus annotées (dataset filtré)


In [ ]:
ann_per_image = df_ann_filtered.groupby('image_id').size().reset_index(name='nb_ann')
top_multi = ann_per_image.nlargest(10, 'nb_ann')
top_multi = top_multi.merge(df_img_filtered[['id', 'file_name']], left_on='image_id', right_on='id')
print(top_multi[['file_name', 'nb_ann']])

### 4.9 Visualisation des boîtes hors limites

On affiche les images problématiques avec leurs boîtes débordantes pour une inspection visuelle. Cette étape permet de comprendre l'origine des anomalies.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
import pandas as pd

# --- 1. Recharger ou s'assurer que df_img et df_ann existent ---
# (si vous les avez déjà, passez cette étape)
# df_img = pd.DataFrame(images)  # avec colonnes id, file_name, width, height
# df_ann = pd.DataFrame(annotations)  # avec colonnes id, image_id, category_id, bbox

# --- 2. Fusionner et détecter les boîtes hors limites ---
# On fusionne avec 'file_name', 'width', 'height'
df_ann_with_img = df_ann.merge(
    df_img[['id', 'file_name', 'width', 'height']],
    left_on='image_id',
    right_on='id'
)

# Vérifier ligne par ligne si la boîte dépasse
def is_out_of_bounds(row):
    x, y, w, h = row['bbox']
    return (x + w > row['width']) or (y + h > row['height'])

df_ann_with_img['out_of_bounds'] = df_ann_with_img.apply(is_out_of_bounds, axis=1)
out_of_bounds = df_ann_with_img[df_ann_with_img['out_of_bounds']].copy()

print(f"Nombre d'annotations hors limites : {len(out_of_bounds)}")
print(out_of_bounds[['file_name', 'bbox', 'width', 'height']].head())

# --- 3. Afficher ces images avec les boîtes ---
image_dir = "/content/ena24"  # à adapter si nécessaire

# Limite d'affichage pour éviter de surcharger (modifiez selon besoin)
n_afficher = min(len(out_of_bounds), 10)  # 10 premières

for idx, (_, row) in enumerate(out_of_bounds.head(n_afficher).iterrows()):
    img_path = os.path.join(image_dir, row['file_name'])

    if not os.path.exists(img_path):
        print(f"Image manquante : {img_path}")
        continue

    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)

    x, y, w, h = row['bbox']
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)

    img_w, img_h = row['width'], row['height']
    depassement = ""
    if x + w > img_w:
        depassement += f" droite +{(x+w - img_w):.1f}px"
    if y + h > img_h:
        depassement += f" bas +{(y+h - img_h):.1f}px"

    ax.set_title(f"{row['file_name']} (dim: {img_w}x{img_h})\nDépassement:{depassement}", fontsize=10)
    plt.tight_layout()
    plt.show()

    if idx >= 4:  # Afficher seulement 5 exemples
        print("Affichage limité aux 5 premières images. Pour tout voir, modifiez 'n_afficher'.")
        break

### 4.10 Nettoyage et rognage des boîtes — Génération du JSON propre

On **rogne** (clip) toutes les boîtes englobantes pour qu'elles restent dans les limites de l'image. Les boîtes qui deviennent vides après rognage sont supprimées. Le résultat est sauvegardé dans `annotations_clean.json`.

> **Stratégie de nettoyage** : `x = clip(x, 0, w_img)`, `w = min(w, w_img - x)`. Toute boîte avec `w ≤ 0` ou `h ≤ 0` après rognage est éliminée.


In [ ]:
import json

# Charger le JSON original
with open("/content/drive/MyDrive/annotations.json", "r") as f:
    data = json.load(f)

# Dictionnaire des dimensions d'images
img_dim = {img["id"]: (img["width"], img["height"]) for img in data["images"]}

# Compter les annotations originales
old_ann_count = len(data["annotations"])

# Nettoyer les annotations (rogner les boîtes)
clean_annotations = []
for ann in data["annotations"]:
    img_id = ann["image_id"]
    w_img, h_img = img_dim[img_id]
    x, y, w, h = ann["bbox"]

    # Rogner la boîte aux bords de l'image
    x = max(0, min(x, w_img))
    y = max(0, min(y, h_img))
    w = min(w, w_img - x)
    h = min(h, h_img - y)

    # Ne conserver que si la boîte a une surface positive
    if w > 0 and h > 0:
        ann["bbox"] = [x, y, w, h]
        clean_annotations.append(ann)

# Remplacer les annotations par la version nettoyée
data["annotations"] = clean_annotations

# (Optionnel) Supprimer les images qui n'ont plus aucune annotation
image_ids_with_ann = set(ann["image_id"] for ann in clean_annotations)
data["images"] = [img for img in data["images"] if img["id"] in image_ids_with_ann]

# Sauvegarder le nouveau JSON
output_path = "/content/drive/MyDrive/annotations_clean.json"
with open(output_path, "w") as f:
    json.dump(data, f)

# Affichage correct des statistiques
print(f"Anciennes annotations : {old_ann_count}")
print(f"Nouvelles annotations : {len(clean_annotations)}")
print(f"Images conservées : {len(data['images'])}")

### 4.11 Vérification finale du JSON nettoyé


In [ ]:
import os
import json

# Dossier des images (celui que vous avez utilisé pour l'extraction)
image_dir = "/content/ena24"   # ou "/content/animaux" selon votre cas

# Fichier JSON propre
json_clean_path = "/content/drive/MyDrive/annotations_clean.json"

# Vérifier que les fichiers existent
print("Images dans", image_dir, ":", len(os.listdir(image_dir)))
with open(json_clean_path, 'r') as f:
    data = json.load(f)
    print("Nombre d'images dans JSON :", len(data['images']))
    print("Nombre d'annotations :", len(data['annotations']))

---

## 5 Préparation du pipeline d'entraînement DETR

### 5.1 Installation des dépendances

Installation des librairies Hugging Face nécessaires : `transformers` (DETR), `torch`, `torchvision`, `datasets` et `accelerate`.


In [ ]:
!pip install transformers torch torchvision datasets accelerate -q

### 5.2 Imports et configuration des chemins

Import de tous les modules nécessaires et définition des chemins vers les fichiers (images, JSON original, JSON nettoyé).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset, random_split
from torchvision.datasets import CocoDetection
from transformers import DetrImageProcessor, DetrForObjectDetection, TrainingArguments, Trainer

# Chemins (à adapter si nécessaire)
image_dir = "/content/ena24"               # dossier contenant toutes les images JPG
json_original = "/content/drive/MyDrive/annotations.json"   # fichier original
json_clean = "/content/drive/MyDrive/annotations_clean.json" # où sauvegarder la version nettoyée

# Vérification
print("Images dans le dossier :", len(os.listdir(image_dir)))

### 5.3 Chargement du processeur et construction du dataset

On charge le `DetrImageProcessor` de Hugging Face et on construit un dataset PyTorch personnalisé (`DetrCocoDataset`) héritant de `CocoDetection`. Ce dataset :
1. Charge les images
2. Formatte les annotations au format COCO
3. Appelle le processeur DETR pour normaliser et encoder les données

> ** Format de sortie** : `(pixel_values, labels)` où `labels` contient les boîtes en coordonnées relatives `[x_center, y_center, width, height]` normalisées entre 0 et 1.


In [ ]:
# Recharger le JSON propre
with open(json_clean, 'r') as f:
    data_clean = json.load(f)

# Récupérer les catégories
categories = data_clean['categories']
id2label = {cat['id']: cat['name'] for cat in categories}
label2id = {v: k for k, v in id2label.items()}
num_classes = len(categories)

# Processor DETR
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

class DetrCocoDataset(CocoDetection):
    def __getitem__(self, idx):
        img, target = super().__getitem__(idx)
        # target : liste de dicts avec 'bbox', 'category_id'
        for ann in target:
            # Ajouter area
            bbox = ann['bbox']
            ann['area'] = bbox[2] * bbox[3]
            # Ajouter iscrowd si absent
            if 'iscrowd' not in ann:
                ann['iscrowd'] = 0
        image_id = self.ids[idx]
        target_dict = {"image_id": image_id, "annotations": target}
        encoding = processor(images=img, annotations=target_dict, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]   # contient boxes, class_labels, area, etc.
        return pixel_values, labels

# Dataset complet
full_dataset = DetrCocoDataset(root=image_dir, annFile=json_clean)
print("Taille du dataset complet :", len(full_dataset))

### 5.4 Normalisation des IDs (conversion en chaînes)

Certaines versions de la bibliothèque COCO requièrent des IDs sous forme de chaînes de caractères. On effectue cette conversion et on resauvegarde le JSON.

>  **Redémarrez le runtime** après cette étape si demandé.


In [ ]:
import json

json_clean = "/content/drive/MyDrive/annotations_clean.json"

with open(json_clean, 'r') as f:
    data = json.load(f)

# Conversion en chaînes
for img in data['images']:
    img['id'] = str(img['id'])
for ann in data['annotations']:
    ann['image_id'] = str(ann['image_id'])

with open(json_clean, 'w') as f:
    json.dump(data, f)

print("IDs convertis en chaînes. Redémarrez le runtime maintenant.")

### 5.5 Classe `AnimalDataset` — Dataset PyTorch personnalisé

On définit une classe `Dataset` PyTorch complète et flexible :
- Chargement des images depuis le disque
- Construction des annotations au format DETR
- Préprocessing via le `DetrImageProcessor`

Cette approche permet un contrôle total sur le pipeline de données, notamment pour les augmentations futures.


In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class AnimalDataset(Dataset):
    def __init__(self, img_ids, img_dict, ann_by_image, processor, image_dir):
        self.img_ids = img_ids
        self.img_dict = img_dict
        self.ann_by_image = ann_by_image
        self.processor = processor
        self.image_dir = image_dir

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.img_dict[img_id]
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")

        annotations = self.ann_by_image.get(img_id, [])
        target = []
        for ann in annotations:
            bbox = ann['bbox']
            area = bbox[2] * bbox[3]
            target.append({
                'bbox': bbox,
                'category_id': ann['category_id'],
                'area': area,
                'iscrowd': 0
            })

        # Encapsuler dans le format attendu par DETR
        coco_annotation = {"image_id": img_id, "annotations": target}
        encoding = self.processor(images=image, annotations=coco_annotation, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]
        return pixel_values, labels

### 5.6 Instanciation du dataset complet

On charge le JSON nettoyé, on construit les dictionnaires de lookup (`img_dict`, `cat_dict`, `ann_by_image`), on filtre les images existantes et on instancie le dataset complet.


In [ ]:
# Recharger le JSON propre (si nécessaire)
with open(json_clean, 'r') as f:
    coco_data = json.load(f)

img_dict = {img['id']: img for img in coco_data['images']}
cat_dict = {cat['id']: cat['name'] for cat in coco_data['categories']}
ann_by_image = {}
for ann in coco_data['annotations']:
    ann_by_image.setdefault(ann['image_id'], []).append(ann)

# Filtrer les images existantes
image_dir = "/content/ena24"
existing_files = set(os.listdir(image_dir))
valid_img_ids = [img_id for img_id, img in img_dict.items() if img['file_name'] in existing_files]

full_dataset = AnimalDataset(valid_img_ids, img_dict, ann_by_image, processor, image_dir)
print("Taille dataset :", len(full_dataset))

### 5.7 Split stratifié Train / Validation / Test

On divise le dataset en trois partitions :
- **Train (70%)** : données d'entraînement
- **Validation (15%)** : suivi de la loss pendant l'entraînement
- **Test (15%)** : évaluation finale non biaisée

Le split est **stratifié** : on utilise la classe majoritaire de chaque image comme critère de stratification afin de garantir une distribution équilibrée dans chaque partition.


In [ ]:
# Recalculer majority_classes avec la nouvelle instance
majority_classes = []
for img_id in full_dataset.img_ids:
    anns = ann_by_image.get(img_id, [])
    if not anns:
        majority_classes.append(-1)
    else:
        cat_ids = [ann['category_id'] for ann in anns]
        majority_classes.append(Counter(cat_ids).most_common(1)[0][0])

from sklearn.model_selection import train_test_split
train_idx, temp_idx = train_test_split(np.arange(len(full_dataset)), test_size=0.3, stratify=majority_classes, random_state=42)
temp_classes = [majority_classes[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=temp_classes, random_state=42)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

### 5.8 Vérification visuelle d'un échantillon du train set

On affiche une image du jeu d'entraînement avec ses boîtes englobantes **décodées** (conversion des coordonnées normalisées DETR → pixels absolus). Cette vérification est essentielle pour s'assurer que le préprocessing est correct.


In [ ]:
pixel_vals, labels = train_dataset[0]
original_idx = train_dataset.indices[0]
img_id = full_dataset.img_ids[original_idx]
img_info = full_dataset.img_dict[img_id]
img_path = os.path.join(image_dir, img_info['file_name'])
img = Image.open(img_path).convert("RGB")
w, h = img.size

boxes = labels['boxes'].cpu().numpy()
class_labels = labels['class_labels'].cpu().numpy()

fig, ax = plt.subplots(1, figsize=(10,8))
ax.imshow(img)
for box, cls in zip(boxes, class_labels):
    xc, yc, bw, bh = box
    x = (xc - bw/2) * w
    y = (yc - bh/2) * h
    rect = patches.Rectangle((x, y), bw*w, bh*h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    cat_name = cat_dict.get(cls, str(cls))
    ax.text(x, y-5, cat_name, color='white', fontsize=8, bbox=dict(facecolor='red', alpha=0.5))
ax.set_title(img_info['file_name'])
plt.show()

### 5.9 Fonction `collate_fn` — Assemblage des batches

DETR nécessite un `collate_fn` personnalisé car les images d'un même batch peuvent avoir des tailles différentes. Le processeur s'occupe du **padding** pour les aligner avant de les empiler dans un tenseur.


In [ ]:
def collate_fn(batch):
    """
    Batch est une liste de tuples (pixel_values, labels)
    retourne un dict avec 'pixel_values' (batch d'images padées) et 'labels' (liste des labels)
    """
    pixel_values = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    # Padding des images pour qu'elles aient toutes la même taille dans le batch
    pixel_values = processor.pad(pixel_values, return_tensors="pt")["pixel_values"]
    return {"pixel_values": pixel_values, "labels": labels}

### 5.10 Création des DataLoaders

On crée trois `DataLoader` (train, validation, test) avec le `collate_fn` personnalisé. La taille de batch (`batch_size=4`) peut être ajustée selon la mémoire GPU disponible.


In [ ]:
from torch.utils.data import DataLoader

batch_size = 4  # adaptez selon votre mémoire GPU

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print("DataLoaders créés avec succès.")

### 5.11 Test de bon fonctionnement du DataLoader

On récupère un premier batch et on vérifie les dimensions et clés pour s'assurer que le pipeline fonctionne correctement avant de lancer l'entraînement.


In [ ]:
# Test sur train_loader
try:
    batch = next(iter(train_loader))
    print("Batch récupéré !")
    print("Clés :", batch.keys())
    print("pixel_values shape :", batch["pixel_values"].shape)  # (batch_size, 3, H, W)
    print("Nombre de labels :", len(batch["labels"]))
    print("Exemple de labels (première image) :", batch["labels"][0].keys())
except Exception as e:
    print("Erreur :", e)

### 5.12 Décodage des coordonnées normalisées

DETR encode les boîtes en **coordonnées relatives** `[x_center, y_center, width, height]` normalisées entre 0 et 1. On montre ici comment les convertir en coordonnées absolues pour la visualisation.


In [ ]:
# Prendre la première image du batch
pixel_vals = batch["pixel_values"][0]  # shape (3, H, W)
labels_img = batch["labels"][0]

# Les boîtes sont normalisées (coordonnées entre 0 et 1)
boxes_norm = labels_img['boxes']  # (N, 4) : [x_center, y_center, width, height] en relatif
# Pour les afficher, il faudrait les multiplier par la taille originale de l'image (dans labels_img['orig_size'])
orig_h, orig_w = labels_img['orig_size'].tolist()
boxes_abs = boxes_norm.clone()
boxes_abs[:, 0] = (boxes_norm[:, 0] - boxes_norm[:, 2]/2) * orig_w   # x_min
boxes_abs[:, 1] = (boxes_norm[:, 1] - boxes_norm[:, 3]/2) * orig_h   # y_min
boxes_abs[:, 2] = boxes_norm[:, 2] * orig_w   # width
boxes_abs[:, 3] = boxes_norm[:, 3] * orig_h   # height

print("Boîtes absolues (x, y, w, h) :\n", boxes_abs)

### 5.13 Recherche et visualisation d'une image multi-objets

On parcourt le train dataset pour trouver une image avec au moins 2 annotations, ce qui permet de vérifier que plusieurs boîtes sont correctement encodées et décodées.


In [ ]:
# Chercher une image dans train_dataset qui a au moins 2 boîtes
multi_idx = None
for i in range(len(train_dataset)):
    _, labels = train_dataset[i]
    if len(labels['boxes']) >= 2:
        multi_idx = i
        break

if multi_idx is not None:
    print(f"Image trouvée à l'index {multi_idx} avec {len(labels['boxes'])} annotations.")
else:
    print("Aucune image avec plusieurs annotations dans le train_dataset.")

if multi_idx is not None:
    pixel_vals, labels = train_dataset[multi_idx]

    original_idx = train_dataset.indices[multi_idx]
    img_id = full_dataset.img_ids[original_idx]
    img_info = full_dataset.img_dict[img_id]
    img_path = os.path.join(image_dir, img_info['file_name'])
    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    boxes_norm = labels['boxes'].cpu().numpy()
    class_labels = labels['class_labels'].cpu().numpy()

    # Conversion en coordonnées absolues
    boxes_abs = []
    for box in boxes_norm:
        xc, yc, bw, bh = box
        x = (xc - bw/2) * w
        y = (yc - bh/2) * h
        w_box = bw * w
        h_box = bh * h
        boxes_abs.append([x, y, w_box, h_box])

    # Affichage
    fig, ax = plt.subplots(1, figsize=(12, 10))
    ax.imshow(img)
    for (x, y, w_box, h_box), cls in zip(boxes_abs, class_labels):
        rect = patches.Rectangle((x, y), w_box, h_box, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        cat_name = cat_dict.get(cls, str(cls))
        ax.text(x, y-5, cat_name, color='white', fontsize=10, bbox=dict(facecolor='red', alpha=0.5))

    ax.set_title(f"{img_info['file_name']} ({len(boxes_abs)} objets)")
    plt.show()

### 5.14 Vérification du format du batch final


In [ ]:
batch = next(iter(train_loader))
print("Forme des images du batch :", batch["pixel_values"].shape)
print("Première image, première boîte normalisée :", batch["labels"][0]['boxes'][0])
print("Classe associée :", batch["labels"][0]['class_labels'][0])

---

## 6 Configuration du modèle et entraînement

### 6.1 Chargement du dictionnaire des catégories


In [ ]:
with open(json_clean, 'r') as f:
    data = json.load(f)
cat_dict = {cat['id']: cat['name'] for cat in data['categories']}
print(cat_dict)

### 6.2 Chargement et adaptation du modèle DETR

On charge le modèle pré-entraîné `facebook/detr-resnet-50` et on **adapte sa tête de classification** au nombre de catégories de notre dataset :

- `num_labels` : nombre de classes (sans compter le fond)
- `id2label` / `label2id` : mappings entre IDs numériques et noms de classes
- `ignore_mismatched_sizes=True` : permet d'ignorer les incompatibilités de taille lors du chargement des poids (nécessaire car la tête finale est redimensionnée)

> ** Transfer Learning** : Le backbone ResNet-50 et le Transformer restent pré-entraînés sur COCO. Seule la tête de classification est réinitialisée avec le bon nombre de sorties.


In [ ]:
from transformers import DetrForObjectDetection, DetrConfig

# Définir la configuration avec le bon nombre de classes
config = DetrConfig.from_pretrained("facebook/detr-resnet-50")
config.num_labels = len(cat_dict)  # 23
config.id2label = {k: v for k, v in cat_dict.items()}
config.label2id = {v: k for k, v in cat_dict.items()}

# Charger le modèle en ignorant les mismatches de taille
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    config=config,
    ignore_mismatched_sizes=True  # crucial pour adapter la tête
)

print("Modèle chargé avec succès.")

### 6.3 Configuration des hyperparamètres d'entraînement

On utilise l'API `Trainer` de Hugging Face pour simplifier la boucle d'entraînement. Les principaux hyperparamètres :

| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| `num_train_epochs` | 2 | Nombre d'époques (à augmenter pour de meilleures performances) |
| `per_device_train_batch_size` | 4 | Taille de batch pour l'entraînement |
| `eval_strategy` | epoch | Évaluation à chaque époque |
| `load_best_model_at_end` | True | Chargement du meilleur modèle en fin d'entraînement |
| `metric_for_best_model` | eval_loss | Critère de sélection du meilleur modèle |


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./detr_results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    remove_unused_columns=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

### 6.4 Vérification de la tête de classification

On vérifie que la tête de classification a bien été adaptée : elle doit avoir `num_classes + 1` sorties (les classes + la classe "pas d'objet").

> ** Exemple attendu** : `Linear(in_features=256, out_features=num_classes+1, bias=True)`


In [ ]:
print("Tête de classification :", model.class_labels_classifier)
# Doit afficher Linear(in_features=256, out_features=24, bias=True)

### 6.5 Lancement de l'entraînement

On lance le fine-tuning avec `trainer.train()`. L'API Trainer gère automatiquement :
- La boucle d'entraînement et d'évaluation
- Le calcul des gradients et la mise à jour des poids
- La sauvegarde des checkpoints
- Le logging des métriques

>  **Durée estimée** : Environ 20-60 min pour 2 epoch selon le GPU et la taille du dataset.


In [ ]:
trainer.train()

---

## 7️⃣ Évaluation, analyse des résultats et sauvegarde

### 7.1 Analyse des courbes d'apprentissage

Après l'entraînement, on extrait et visualise les courbes de **loss d'entraînement et de validation** pour détecter :
- Le **surapprentissage** (overfitting) : train loss ↓ mais val loss ↑
- Le **sous-apprentissage** (underfitting) : les deux losses restent élevées
- Une bonne **convergence** : les deux losses diminuent et se stabilisent


In [ ]:
# Après trainer.train()
logs = trainer.state.log_history

# Extraire les pertes d'entraînement et de validation
train_losses = [log['loss'] for log in logs if 'loss' in log]
eval_losses = [log['eval_loss'] for log in logs if 'eval_loss' in log]

print(f"Nombre d'époques d'entraînement : {len(train_losses)}")
print(f"Meilleure perte de validation : {min(eval_losses):.4f}")

# Tracer l'évolution des pertes
import matplotlib.pyplot as plt
plt.plot(train_losses, label='Train loss')
plt.plot(eval_losses, label='Val loss')
plt.xlabel('Époque')
plt.ylabel('Perte')
plt.legend()
plt.title('Courbes d’apprentissage')
plt.show()

### 7.2 Sauvegarde du modèle et du processeur

On sauvegarde le modèle fine-tuné et son processeur dans Google Drive pour une utilisation ultérieure (inférence, déploiement, reprise d'entraînement).


In [ ]:
# Sauvegarder le modèle et le processeur
model_save_path = "/content/drive/MyDrive/detr_animal_model"
model.save_pretrained(model_save_path)
processor.save_pretrained(model_save_path)
print(f"Modèle sauvegardé dans {model_save_path}")

### 7.3 Chargement du meilleur checkpoint pour l'inférence

On charge le meilleur checkpoint sauvegardé par le `Trainer`. La numérotation des checkpoints correspond au nombre de steps d'entraînement effectués.

> ** Conseil** : Si `load_best_model_at_end=True`, le meilleur modèle est automatiquement rechargé à la fin de `trainer.train()`. Cette cellule est utile pour recharger le modèle dans une session future.


In [ ]:
from transformers import DetrForObjectDetection

best_model_path = "./detr_results/checkpoint-XXX"  # Remplacez XXX par le numéro du meilleur checkpoint (ex: 500)
# Ou trouvez automatiquement le dernier checkpoint :
import glob
checkpoints = glob.glob("./detr_results/checkpoint-*")
best_checkpoint = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]  # dernier
print(f"Chargement du checkpoint : {best_checkpoint}")
model = DetrForObjectDetection.from_pretrained(best_checkpoint)
model.to(device)

---

##  Conclusion et prochaines étapes

### Résumé du pipeline

Ce notebook a couvert l'intégralité du pipeline de fine-tuning DETR :

1.  **Chargement et exploration** du dataset au format COCO
2.  **EDA statistique** : distribution des classes, des boîtes, des tailles
3.  **Nettoyage** : suppression des doublons, correction des boîtes invalides
4.  **Dataset PyTorch** personnalisé avec split stratifié 70/15/15
5.  **Fine-tuning** de `facebook/detr-resnet-50` avec l'API Trainer
6.  **Sauvegarde** du modèle et des checkpoints

### Pistes d'amélioration

-  **Augmenter le nombre d'époques** (5-20 selon la convergence)
-  **Augmentation de données** : flip horizontal, recadrage aléatoire, jitter de couleur
-  **Rééchantillonnage** pour les classes sous-représentées
-  **Métriques COCO** : implémenter le calcul du mAP (mean Average Precision)
-  **Inférence sur nouvelles images** : pipeline complet de post-traitement



In [ ]:
import nbformat

nb = nbformat.read("DETR (DEtection TRansformer).ipynb", as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, "DETR (DEtection TRansformer).ipynb")